## librerias

In [33]:
import os
import webbrowser
import pandas as pd
import json
import geopandas as gpd
import colorsys
import numpy as np
import http.server
import socketserver
from threading import Thread
import time
import hashlib
import unicodedata

## bases 

In [34]:
usuario = os.getlogin()

In [35]:
base = pd.read_excel(fr"C:\Users\{usuario}\Downloads\UnidadesIMB_CS!_v2.xlsx",sheet_name="Sheet 1")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

## back

In [36]:
base = base.drop(columns=['poblacion_sin_dh_menos_30', 'poblacion_con_dh_menos_30', 'pob_imo_pct'])


base.columns = (
    base.columns
        .str.strip()
        .str.lower()
        .map(
            lambda x: ''.join(
                c for c in unicodedata.normalize('NFD', x)
                if unicodedata.category(c) != 'Mn'
            )
        )
        .str.replace(" ", "_", regex=False)
)

In [37]:
base = base.merge(
    clues[["clues_imb", "entidad"]],
    on="clues_imb",
    how="left"
)

In [38]:
# base.columns

In [39]:
# Obtener entidades únicas
entidades_unicas = sorted(base['entidad'].dropna().unique())

# Identificar columnas de equipamiento (todas excepto las que no son numéricas)
columnas_excluir = ['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada', 'entidad']
columnas_equipamiento = [col for col in base.columns if col not in columnas_excluir]

# Obtener unidades por entidad
def get_unidades_por_entidad(entidad):
    """Obtiene todas las unidades de una entidad con sus datos"""
    df_entidad = base[base['entidad'] == entidad]
    return df_entidad.to_dict('records')

In [40]:
# Configuración de colores
COLOR_PRIMARIO = "#FAF2F5"
COLOR_SECUNDARIO = '#AE8640'
COLOR_HBC = "#FDFDFDC0"
COLOR_FONDO = "#235B4E"
COLOR_BORDE = '#7A1737'
COLOR_TEXTO = '#000000'

In [43]:
# Función simple para formatear nombres de columnas
def formatear_nombre(columna):
    nombre = columna.replace('_', ' ')
    nombre = nombre.split('.')[0]
    palabras = nombre.split()
    palabras = [p.capitalize() for p in palabras]
    return ' '.join(palabras)


## front

In [44]:


script_url = "https://script.google.com/macros/s/AKfycbzVpuRnysY0btUZryqx2vetKiRisDV-53QXLsQ6TncG9iRwVNB77e1MD11C9U4QXCQ5/exec"

# Obtener columnas de equipamiento
columnas_equipamiento = [col for col in base.columns if col not in ['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada', 'entidad']]

html_content = f'''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Cuestionario de Equipamiento - IMSS Bienestar</title>
    <link href="https://fonts.googleapis.com/css2?family=League+Spartan:wght@400;500;600;700&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: "League Spartan", sans-serif;
        }}

        body {{
            background: linear-gradient(135deg, {COLOR_PRIMARIO} 0%, #fff 100%);
            min-height: 100vh;
            padding: 20px;
        }}

        .container {{
            width: 100%;
            max-width: 1400px;
            margin: 0 auto;
        }}

        /* Menu Hamburguesa */
        .menu-container {{
            position: fixed;
            top: 20px;
            right: 20px;
            z-index: 1001;
        }}

        .menu-btn {{
            background: {COLOR_FONDO};
            border: none;
            border-radius: 50%;
            width: 50px;
            height: 50px;
            cursor: pointer;
            display: flex;
            flex-direction: column;
            justify-content: center;
            align-items: center;
            gap: 6px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            transition: 0.3s;
        }}

        .menu-btn:hover {{
            background: {COLOR_SECUNDARIO};
            transform: scale(1.05);
        }}

        .menu-btn span {{
            width: 25px;
            height: 3px;
            background: white;
            border-radius: 3px;
            transition: 0.3s;
        }}

        .menu-panel {{
            position: fixed;
            top: 0;
            right: -400px;
            width: 380px;
            height: 100%;
            background: white;
            box-shadow: -2px 0 10px rgba(0,0,0,0.1);
            z-index: 1002;
            transition: 0.3s;
            overflow-y: auto;
            padding: 80px 25px 25px 25px;
        }}

        .menu-panel.active {{
            right: 0;
        }}

        .menu-overlay {{
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.5);
            z-index: 1001;
            display: none;
        }}

        .menu-overlay.active {{
            display: block;
        }}

        .menu-panel h2 {{
            color: {COLOR_FONDO};
            margin-bottom: 20px;
            font-size: 24px;
            border-bottom: 3px solid {COLOR_SECUNDARIO};
            padding-bottom: 10px;
        }}

        .menu-panel h3 {{
            color: {COLOR_SECUNDARIO};
            margin: 20px 0 10px 0;
            font-size: 18px;
        }}

        .menu-panel p {{
            color: #333;
            line-height: 1.6;
            margin-bottom: 15px;
        }}

        .menu-panel ul, .menu-panel ol {{
            color: #555;
            margin-left: 20px;
            margin-bottom: 15px;
        }}

        .menu-panel li {{
            margin-bottom: 8px;
        }}

        .close-menu {{
            position: absolute;
            top: 20px;
            right: 20px;
            background: none;
            border: none;
            font-size: 30px;
            cursor: pointer;
            color: {COLOR_FONDO};
        }}

        .header {{
            background: {COLOR_FONDO};
            padding: 20px;
            color: white;
            margin-bottom: 30px;
            border-radius: 15px;
            text-align: center;
            position: relative;
        }}

        .header img {{
            height: 50px;
            margin-bottom: 10px;
        }}

        .header h1 {{
            font-size: 24px;
            margin-bottom: 5px;
        }}

        /* Instrucciones rapidas */
        .instrucciones-rapidas {{
            background: white;
            border-radius: 12px;
            padding: 15px 20px;
            margin-bottom: 20px;
            display: flex;
            justify-content: center;
            gap: 30px;
            flex-wrap: wrap;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}

        .instruccion-item {{
            display: flex;
            align-items: center;
            gap: 12px;
            font-size: 14px;
            font-weight: 500;
        }}

        .instruccion-color {{
            width: 24px;
            height: 24px;
            border-radius: 6px;
        }}

        .color-verde {{
            background: #e8f5e9;
            border: 2px solid #2e7d32;
        }}

        .color-rojo {{
            background: #ffebee;
            border: 2px solid #c62828;
        }}

        .instruccion-texto {{
            color: #333;
        }}

        .instruccion-texto strong {{
            color: {COLOR_FONDO};
        }}

        .estados-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(180px, 1fr));
            gap: 15px;
            max-height: 600px;
            overflow-y: auto;
            padding: 10px;
        }}

        .estado-btn {{
            background: {COLOR_FONDO};
            border: none;
            border-radius: 12px;
            padding: 15px 10px;
            color: white;
            font-weight: 600;
            cursor: pointer;
            transition: 0.3s;
            font-size: 14px;
        }}

        .estado-btn:hover {{
            background: {COLOR_SECUNDARIO};
            transform: translateY(-3px);
        }}

        .modal {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.8);
            backdrop-filter: blur(5px);
            justify-content: center;
            align-items: center;
            z-index: 1000;
        }}

        .modal.active {{
            display: flex;
        }}
        
        .modal-form.active {{
            display: flex;
        }}

        .modal-content {{
            position: relative;
            background: linear-gradient(135deg, {COLOR_BORDE}dd);
            backdrop-filter: none;
            border-radius: 20px;
            padding: 30px;
            width: 90%;
            max-width: 450px;
            color: white;
            border: 1px solid rgba(255,255,255,0.2);
            animation: slideUp 0.3s;
        }}

        @keyframes slideUp {{
            from {{ transform: translateY(20px); opacity: 0; }}
            to {{ transform: translateY(0); opacity: 1; }}
        }}

        .modal h2 {{
            text-align: center;
            margin-bottom: 10px;
            font-size: 28px;
        }}

        .modal p {{
            text-align: center;
            margin-bottom: 20px;
            opacity: 0.9;
        }}

        .close-btn {{
            position: absolute;
            top: 10px;
            right: 15px;
            background: none;
            border: none;
            font-size: 28px;
            cursor: pointer;
            color: white;
            opacity: 0.8;
            transition: opacity 0.2s;
            line-height: 1;
        }}

        .close-btn:hover {{
            opacity: 1;
        }}

        .modal-form {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.5);
            backdrop-filter: blur(5px);
            justify-content: center;
            align-items: center;
            z-index: 1000;
        }}

        .modal-form.active {{
            display: flex;
        }}

        .modal-form-content {{
            position: relative;
            background: white;
            border-radius: 20px;
            padding: 30px;
            width: 95%;
            max-width: 1400px;
            max-height: 90vh;
            overflow-y: auto;
        }}

        .modal-form-content .close-btn {{
            color: #333;
            top: 10px;
            right: 15px;
        }}

        .form-group {{
            margin-bottom: 15px;
        }}

        .form-group label {{
            display: block;
            margin-bottom: 5px;
            font-weight: 600;
        }}

        .form-group input, .form-group select {{
            width: 100%;
            padding: 10px;
            background: rgba(255,255,255,0.2);
            border: 1px solid rgba(255,255,255,0.3);
            border-radius: 8px;
            color: white;
            font-size: 14px;
        }}

        .form-group input::placeholder {{
            color: rgba(255,255,255,0.6);
        }}

        .form-group input:focus {{
            outline: none;
            border-color: white;
            background: rgba(255,255,255,0.25);
        }}

        .help-text {{
            font-size: 12px;
            margin-top: 5px;
            opacity: 0.8;
        }}

        .btn {{
            width: 100%;
            padding: 12px;
            background: white;
            color: {COLOR_FONDO};
            border: none;
            border-radius: 8px;
            font-weight: 700;
            font-size: 16px;
            cursor: pointer;
            transition: 0.3s;
            margin-top: 10px;
        }}

        .btn:hover {{
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.3);
        }}

        .error {{
            color: #ffcccc;
            font-size: 13px;
            margin-top: 10px;
            text-align: center;
            display: none;
        }}

        .unidades-table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 20px;
        }}

        .unidades-table th, .unidades-table td {{
            border: 1px solid #ddd;
            padding: 8px;
            text-align: left;
            vertical-align: middle;
        }}

        .unidades-table th {{
            background-color: {COLOR_FONDO};
            color: white;
            position: sticky;
            top: 0;
        }}

        .unidades-table tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}

        .campo-container {{
            display: flex;
            gap: 8px;
            align-items: center;
            flex-wrap: wrap;
        }}

        .valor-mostrado {{
            display: inline-block;
            background: #e8f5e9;
            color: #2e7d32;
            padding: 5px 10px;
            border-radius: 5px;
            font-weight: bold;
            font-size: 14px;
            min-width: 50px;
            text-align: center;
            cursor: pointer;
            transition: 0.2s;
        }}

        .valor-mostrado:hover {{
            background: #c8e6c9;
            transform: scale(1.02);
        }}

        .valor-vacio {{
            background: #ffebee;
            color: #c62828;
            cursor: pointer;
        }}

        .valor-vacio:hover {{
            background: #ffcdd2;
        }}

        .input-edicion {{
            width: 80px;
            padding: 5px;
            border: 2px solid {COLOR_SECUNDARIO};
            border-radius: 4px;
            text-align: center;
            font-size: 14px;
        }}

        .btn-accion {{
            color: white;
            border: none;
            padding: 5px 10px;
            border-radius: 5px;
            cursor: pointer;
            font-size: 12px;
            font-weight: 600;
            transition: 0.3s;
            white-space: nowrap;
        }}

        .btn-accion {{
            background: #2196F3;
        }}

        .btn-accion:hover {{
            background: #0b7dda;
        }}

        .table-container {{
            overflow-x: auto;
            max-height: 60vh;
            overflow-y: auto;
        }}

        .btn-guardar-todo {{
            background: {COLOR_SECUNDARIO};
            color: white;
            margin-right: 10px;
        }}

        .btn-cancelar {{
            background: #666;
            color: white;
        }}

        .acciones {{
            display: flex;
            gap: 10px;
            margin-top: 20px;
            justify-content: center;
        }}

        .progress {{
            margin-bottom: 20px;
            padding: 10px;
            background: #f0f0f0;
            border-radius: 8px;
            color: #333;
        }}

        .badge {{
            display: inline-block;
            padding: 3px 8px;
            border-radius: 12px;
            font-size: 11px;
            font-weight: bold;
        }}
        
        .badge-success {{
            background: #4CAF50;
            color: white;
        }}
        
        .badge-warning {{
            background: #ff9800;
            color: white;
        }}

        /* Paginacion */
        .pagination {{
            display: flex;
            justify-content: center;
            align-items: center;
            gap: 10px;
            margin-top: 20px;
            margin-bottom: 20px;
        }}

        .pagination button {{
            background: {COLOR_FONDO};
            color: white;
            border: none;
            padding: 8px 16px;
            border-radius: 8px;
            cursor: pointer;
            transition: 0.3s;
            font-weight: 600;
        }}

        .pagination button:hover:not(:disabled) {{
            background: {COLOR_SECUNDARIO};
            transform: translateY(-2px);
        }}

        .pagination button:disabled {{
            opacity: 0.5;
            cursor: not-allowed;
        }}

        .pagination span {{
            font-size: 14px;
            font-weight: 600;
        }}

        .page-info {{
            background: {COLOR_FONDO};
            color: white !important;
            padding: 5px 12px;
            border-radius: 20px;
            font-size: 14px;
        }}
    </style>
</head>
<body>
    <!-- Menu Hamburguesa -->
    <div class="menu-container">
        <button class="menu-btn" onclick="toggleMenu()">
            <span></span>
            <span></span>
            <span></span>
        </button>
    </div>

    <div class="menu-overlay" id="menuOverlay" onclick="toggleMenu()"></div>
    
    <div class="menu-panel" id="menuPanel">
        <button class="close-menu" onclick="toggleMenu()">&times;</button>
        <h2>Instrucciones</h2>
        
        <h3>1. Seleccionar Estado</h3>
        <p>Haga clic en el boton del estado correspondiente para comenzar el registro de equipamiento.</p>
        
        <h3>2. Registrar Datos del Usuario</h3>
        <p>Complete el formulario con:</p>
        <ul>
            <li>Nombre completo (nombre(s) y apellidos)</li>
            <li>Correo electronico institucional</li>
        </ul>
        
        <h3>3. Llenar Equipamiento</h3>
        <p>Se muestran 5 unidades medicas por pagina. Use los botones de navegacion para avanzar.</p>
        <p><strong style="color:#2e7d32">VERDE:</strong> Valor ya registrado - Haga clic para modificar</p>
        <p><strong style="color:#c62828">ROJO:</strong> Campo pendiente - Haga clic para llenar</p>
        
        <h3>4. Guardar Informacion</h3>
        <p>Una vez completado todo, presione el boton <strong>"Guardar todo"</strong> para enviar los datos.</p>
        
        <h3>Progreso</h3>
        <p>La barra de progreso muestra el avance del llenado de todas las unidades.</p>
        
        <h3>Consejos</h3>
        <ul>
            <li>Use solo numeros enteros</li>
            <li>Los datos se mantienen al cambiar de pagina</li>
            <li>Haga clic en cualquier valor para editarlo</li>
        </ul>
        
        <h3>Soporte</h3>
        <p>Si tiene problemas, contacte al area de sistemas de IMSS Bienestar.</p>
    </div>

    <div class="container">
        <div class="header">
            <img src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" alt="IMSS Bienestar">
            <h1>CUESTIONARIO DE EQUIPAMIENTO POR UNIDAD MEDICA</h1>
            <p>Seleccione una entidad para registrar el equipamiento de sus unidades</p>
        </div>

        <div class="estados-grid" id="estadosGrid">
'''

# Generar botones de entidades
for entidad in entidades_unicas:
    num_unidades = len(base[base['entidad'] == entidad])
    html_content += f'''
            <button class="estado-btn" onclick="abrirModal('{entidad}')">
                {entidad}<br>
                <small style="font-size: 11px;">{num_unidades} unidades</small>
            </button>
    '''

html_content += f'''
        </div>
    </div>

    <!-- Modal de registro de usuario -->
    <div class="modal" id="loginModal">
        <div class="modal-content">
            <button class="close-btn" onclick="cerrarModal()">&times;</button>
            <h2 id="modalEstado"></h2>
            <p>Registre sus datos para continuar</p>
            
            <div class="form-group">
                <label>Entidad</label>
                <input type="text" id="usuarioInput" readonly>
            </div>
            
            <div class="form-group">
                <label>Nombre completo</label>
                <input type="text" id="nombreInput" placeholder="Ej: Juan Carlos Perez Gonzalez" required>
                <div class="help-text">Ingrese nombre(s) y apellidos completos</div>
            </div>
            
            <div class="form-group">
                <label>Correo electronico institucional</label>
                <input type="email" id="emailInput" placeholder="ejemplo@imssbienestar.gob.mx" required>
                <div class="help-text">Ingrese su correo electronico</div>
            </div>
            
            <button class="btn" onclick="validarDatos()">Continuar</button>
            <div id="errorMsg" class="error"></div>
        </div>
    </div>

    <!-- Modal del formulario -->
    <div class="modal-form" id="formModal">
        <div class="modal-form-content">
            <button class="close-btn" onclick="cerrarFormModal()">&times;</button>
            <h2 id="formEstado"></h2>
            <div id="userInfo" style="background: #f0f0f0; padding: 10px; border-radius: 8px; margin-bottom: 15px; font-size: 14px;"></div>
            <div id="progressInfo" class="progress"></div>
            
            <!-- Instrucciones rapidas -->
            <div class="instrucciones-rapidas">
                <div class="instruccion-item">
                    <div class="instruccion-color color-verde"></div>
                    <div class="instruccion-texto"><strong>VERDE</strong>  Haga clic para MODIFICAR</div>
                </div>
                <div class="instruccion-item">
                    <div class="instruccion-color color-rojo"></div>
                    <div class="instruccion-texto"><strong>ROJO</strong>  Haga clic para LLENAR</div>
                </div>
            </div>
            
            <div class="table-container">
                <table class="unidades-table" id="unidadesTable">
                    <thead>
                        <tr>
                            <th style="min-width: 100px;">CLUES</th>
                            <th style="min-width: 250px;">Unidad Medica</th>
                            <th style="min-width: 150px;">Categoria</th>
                            {''.join([f'<th style="min-width: 180px;">{formatear_nombre(col)}</th>' for col in columnas_equipamiento])}
                        </tr>
                    </thead>
                    <tbody id="tableBody">
                    </tbody>
                </table>
            </div>
            
            <!-- Paginacion -->
            <div class="pagination" id="pagination">
                <button onclick="cambiarPagina(-1)" id="btnAnterior">Anterior</button>
                <span id="paginaInfo"></span>
                <button onclick="cambiarPagina(1)" id="btnSiguiente">Siguiente</button>
            </div>
            
            <div class="acciones">
                <button type="button" class="btn btn-guardar-todo" onclick="guardarFormulario()">Guardar todo</button>
                <button type="button" class="btn btn-cancelar" onclick="cerrarFormModal()">Cancelar</button>
            </div>
        </div>
    </div>

    <script>
        // Funcion para el menu hamburguesa
        function toggleMenu() {{
            const panel = document.getElementById('menuPanel');
            const overlay = document.getElementById('menuOverlay');
            panel.classList.toggle('active');
            overlay.classList.toggle('active');
        }}

        // Cerrar menu con Escape
        document.addEventListener('keydown', function(e) {{
            if (e.key === 'Escape') {{
                const panel = document.getElementById('menuPanel');
                const overlay = document.getElementById('menuOverlay');
                panel.classList.remove('active');
                overlay.classList.remove('active');
                cerrarModal();
                cerrarFormModal();
            }}
        }});

        // Datos completos de todas las unidades
        const datosUnidades = {json.dumps({entidad: get_unidades_por_entidad(entidad) for entidad in entidades_unicas}, ensure_ascii=False)};
        
        // Columnas de equipamiento
        const columnasEquipamiento = {json.dumps(columnas_equipamiento)};
        
        let estadoSeleccionado = '';
        let datosActuales = [];
        let usuarioActual = {{
            nombre: '',
            email: '',
            entidad: ''
        }};
        
        // Variables de paginacion
        let paginaActual = 0;
        let unidadesPorPagina = 5;
        let totalPaginas = 0;

        function abrirModal(estado) {{
            estadoSeleccionado = estado;
            document.getElementById('modalEstado').textContent = estado;
            document.getElementById('usuarioInput').value = estado;
            document.getElementById('nombreInput').value = '';
            document.getElementById('emailInput').value = '';
            document.getElementById('errorMsg').style.display = 'none';
            document.getElementById('loginModal').classList.add('active');
            document.getElementById('nombreInput').focus();
        }}

        function cerrarModal() {{
            document.getElementById('loginModal').classList.remove('active');
        }}

        function validarNombreCompleto(nombre) {{
            const nombreTrim = nombre.trim();
            const partes = nombreTrim.split(/\\s+/);
            if (partes.length < 2) return false;
            if (partes.length > 5) return false;
            for (let parte of partes) {{
                if (parte.length < 2) return false;
            }}
            return true;
        }}

        function validarDatos() {{
            const nombre = document.getElementById('nombreInput').value;
            const email = document.getElementById('emailInput').value.trim();
            
            if (!validarNombreCompleto(nombre)) {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese su nombre completo (nombre(s) y apellidos)';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            if (email === '') {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese un correo electronico';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            
            const emailRegex = /^[^\\s@]+@([^\\s@]+\\.)+[^\\s@]+$/;
            if (!emailRegex.test(email)) {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese un correo electronico valido';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            
            usuarioActual = {{
                nombre: nombre.trim(),
                email: email,
                entidad: estadoSeleccionado
            }};
            
            document.getElementById('errorMsg').style.display = 'none';
            cerrarModal();
            abrirFormulario(estadoSeleccionado);
        }}

        function abrirFormulario(estado) {{
            document.getElementById('formEstado').innerHTML = `Cuestionario de Equipamiento - <strong>${{estado}}</strong>`;
            document.getElementById('userInfo').innerHTML = `
                <strong>Registrado por:</strong> ${{usuarioActual.nombre}} | 
                <strong>Correo:</strong> ${{usuarioActual.email}} | 
                <strong>Entidad:</strong> ${{usuarioActual.entidad}}
            `;
            document.getElementById('formModal').classList.add('active');
            cargarUnidades(estado);
        }}

        function crearCampoEquipamiento(valorActual, filaOriginal, columna) {{
            const container = document.createElement('div');
            container.className = 'campo-container';
            
            const valorMostrado = document.createElement('div');
            valorMostrado.className = 'valor-mostrado';
            
            const tieneValor = (valorActual !== null && valorActual !== undefined && !isNaN(valorActual) && valorActual !== 0 && valorActual !== '');
            const valorDisplay = tieneValor ? valorActual : 'PENDIENTE';
            
            valorMostrado.textContent = valorDisplay;
            
            if (!tieneValor) {{
                valorMostrado.classList.add('valor-vacio');
            }}
            
            let estadoEdicion = false;
            let inputField = null;
            let btnAccion = null;
            let valorOriginal = tieneValor ? valorActual : null;
            
            const iniciarEdicion = () => {{
                if (estadoEdicion) return;
                
                estadoEdicion = true;
                container.innerHTML = '';
                
                // Crear input
                inputField = document.createElement('input');
                inputField.type = 'number';
                inputField.step = '1';
                inputField.value = (tieneValor && valorActual !== null) ? valorActual : '';
                inputField.placeholder = '0';
                inputField.className = 'input-edicion';
                
                // Crear boton de accion
                btnAccion = document.createElement('button');
                btnAccion.className = 'btn-accion';
                btnAccion.textContent = 'ACEPTAR';
                
                // Funcion para guardar o aceptar
                const finalizarEdicion = () => {{
                    const nuevoValor = inputField.value;
                    const haCambiado = nuevoValor !== String(valorOriginal);
                    
                    if (haCambiado) {{
                        // Hubo modificacion - GUARDAR
                        let numero = parseInt(nuevoValor);
                        if (nuevoValor === '') {{
                            datosActuales[filaOriginal][columna] = null;
                        }} else if (!isNaN(numero) && numero >= 0) {{
                            datosActuales[filaOriginal][columna] = numero;
                        }} else {{
                            alert('Por favor, ingrese un numero valido mayor o igual a 0');
                            return;
                        }}
                    }}
                    // Si no hubo cambio, simplemente aceptar sin guardar
                    
                    actualizarProgreso();
                    mostrarPagina(); // Recargar la pagina
                }};
                
                btnAccion.onclick = finalizarEdicion;
                
                // Enter para confirmar
                inputField.onkeypress = (e) => {{
                    if (e.key === 'Enter') {{
                        finalizarEdicion();
                    }}
                }};
                
                container.appendChild(inputField);
                container.appendChild(btnAccion);
                inputField.focus();
            }};
            
            valorMostrado.onclick = iniciarEdicion;
            container.appendChild(valorMostrado);
            
            return container;
        }}

        function cargarUnidades(estado) {{
            const unidades = datosUnidades[estado] || [];
            datosActuales = JSON.parse(JSON.stringify(unidades));
            
            totalPaginas = Math.ceil(datosActuales.length / unidadesPorPagina);
            paginaActual = 0;
            
            if (totalPaginas > 0) {{
                mostrarPagina();
            }}
            actualizarProgreso();
        }}
        
        function mostrarPagina() {{
            const inicio = paginaActual * unidadesPorPagina;
            const fin = inicio + unidadesPorPagina;
            const unidadesPagina = datosActuales.slice(inicio, fin);
            
            const tbody = document.getElementById('tableBody');
            tbody.innerHTML = '';
            
            unidadesPagina.forEach((unidad, idx) => {{
                const filaOriginal = inicio + idx;
                const row = tbody.insertRow();
                
                const cellClues = row.insertCell(0);
                cellClues.innerHTML = `<strong>${{unidad.clues_imb || ''}}</strong>`;
                cellClues.style.backgroundColor = '#f0f0f0';
                
                const cellNombre = row.insertCell(1);
                cellNombre.innerHTML = unidad.nombre_de_la_unidad || '';
                cellNombre.style.backgroundColor = '#f0f0f0';
                
                const cellCategoria = row.insertCell(2);
                cellCategoria.innerHTML = unidad.categoria_gerencial_ampliada || '';
                cellCategoria.style.backgroundColor = '#f0f0f0';
                
                columnasEquipamiento.forEach(col => {{
                    const cell = row.insertCell();
                    const valorActual = unidad[col];
                    const campo = crearCampoEquipamiento(valorActual, filaOriginal, col);
                    cell.appendChild(campo);
                }});
            }});
            
            // Actualizar informacion de paginacion
            const desde = inicio + 1;
            const hasta = Math.min(fin, datosActuales.length);
            document.getElementById('paginaInfo').innerHTML = `<span class="page-info">Pagina ${{paginaActual + 1}} de ${{totalPaginas}} | Mostrando ${{desde}} - ${{hasta}} de ${{datosActuales.length}} unidades</span>`;
            
            // Actualizar estado de botones
            document.getElementById('btnAnterior').disabled = paginaActual === 0;
            document.getElementById('btnSiguiente').disabled = paginaActual === totalPaginas - 1;
        }}
        
        function cambiarPagina(direccion) {{
            const nuevaPagina = paginaActual + direccion;
            if (nuevaPagina >= 0 && nuevaPagina < totalPaginas) {{
                paginaActual = nuevaPagina;
                mostrarPagina();
            }}
        }}
        
        function actualizarProgreso() {{
            let completados = 0;
            let totalCampos = 0;
            
            datosActuales.forEach(unidad => {{
                columnasEquipamiento.forEach(col => {{
                    totalCampos++;
                    const valor = unidad[col];
                    if (valor !== null && valor !== undefined && !isNaN(valor) && valor !== 0 && valor !== '') {{
                        completados++;
                    }}
                }});
            }});
            
            const porcentaje = totalCampos > 0 ? Math.round((completados / totalCampos) * 100) : 0;
            document.getElementById('progressInfo').innerHTML = `
                <strong>Progreso de llenado:</strong>
                <div style="background: #ddd; border-radius: 10px; margin-top: 5px;">
                    <div style="background: {COLOR_SECUNDARIO}; width: ${{porcentaje}}%; height: 20px; border-radius: 10px; transition: width 0.3s;"></div>
                </div>
                <p style="margin-top: 5px;">${{completados}} de ${{totalCampos}} campos completados (${{porcentaje}}%)</p>
                <span class="badge ${{porcentaje === 100 ? 'badge-success' : 'badge-warning'}}">
                    ${{porcentaje === 100 ? 'COMPLETO' : 'PENDIENTE'}}
                </span>
            `;
        }}

        function cerrarFormModal() {{
            if (confirm('Cancelar edicion? Los cambios no guardados se perderan.')) {{
                document.getElementById('formModal').classList.remove('active');
            }}
        }}

        function guardarFormulario() {{
            const datosAGuardar = {{
                entidad: estadoSeleccionado,
                fecha_registro: new Date().toISOString(),
                usuario: {{
                    nombre: usuarioActual.nombre,
                    email: usuarioActual.email,
                    entidad: usuarioActual.entidad
                }},
                unidades: datosActuales.map(unidad => {{
                    const datosUnidad = {{
                        clues_imb: unidad.clues_imb,
                        nombre_de_la_unidad: unidad.nombre_de_la_unidad,
                        categoria: unidad.categoria_gerencial_ampliada
                    }};
                    
                    columnasEquipamiento.forEach(col => {{
                        const valor = unidad[col];
                        if (valor !== null && valor !== undefined && !isNaN(valor) && valor !== 0 && valor !== '') {{
                            datosUnidad[col] = valor;
                        }}
                    }});
                    
                    return datosUnidad;
                }})
            }};
            
            console.log('Datos a guardar:', JSON.stringify(datosAGuardar, null, 2));
            
            if (confirm(`Guardar datos?\\n\\nEntidad: ${{estadoSeleccionado}}\\nRegistrado por: ${{usuarioActual.nombre}}\\nUnidades: ${{datosAGuardar.unidades.length}}\\nProgreso: ${{document.getElementById('progressInfo').innerText.match(/\\d+%/)[0]}}`)) {{
                enviarDatos(datosAGuardar);
            }}
        }}

        function enviarDatos(datos) {{
            const scriptURL = '{script_url}';
            
            fetch(scriptURL, {{ 
                method: 'POST', 
                mode: 'cors',
                headers: {{
                    'Content-Type': 'application/json',
                }},
                body: JSON.stringify(datos)
            }})
            .then(response => response.json())
            .then(data => {{
                console.log('Respuesta:', data);
                alert('Datos enviados con exito!');
                cerrarFormModal();
            }})
            .catch(error => {{
                console.error('Error:', error);
                alert('Error al enviar datos. Los datos se guardaron en la consola (F12).');
            }});
        }}

        // Limpiar errores al escribir
        document.getElementById('nombreInput').addEventListener('input', function() {{
            this.style.borderColor = 'rgba(255,255,255,0.3)';
            document.getElementById('errorMsg').style.display = 'none';
        }});
        
        document.getElementById('emailInput').addEventListener('input', function() {{
            this.style.borderColor = 'rgba(255,255,255,0.3)';
            document.getElementById('errorMsg').style.display = 'none';
        }});

        // Cerrar modales al hacer clic fuera
        document.getElementById('loginModal').addEventListener('click', function(e) {{
            if (e.target === this) {{
                cerrarModal();
            }}
        }});

        // Enter para continuar
        document.getElementById('emailInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                validarDatos();
            }}
        }});
        
        document.getElementById('nombreInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                document.getElementById('emailInput').focus();
            }}
        }});
    </script>
</body>
</html>
'''



## salida

In [45]:
# Guardar el archivo HTML
usuario = os.getlogin()
destino_html = fr"C:\Users\{usuario}\Downloads\formu\formulario_an\index.html"

try:
    with open(destino_html, "w", encoding="utf-8") as file:
        file.write(html_content)
    
   
    
    webbrowser.open(destino_html)
    
except Exception as e:
    
    destino_actual = os.path.join(os.getcwd(), "cuestionario_equipamiento_imss.html")
    with open(destino_actual, "w", encoding="utf-8") as file:
        file.write(html_content)
 
    webbrowser.open(destino_actual)